# 03 · LightGBM matcher: which bucket candidates are real matches?

Notebook 02 gave every S1 a bucket of its most similar S2/S3 records. This notebook trains
LightGBM to decide which candidates in each bucket are really the same business.

**Input:**
* one sample folder written by `02_e5_embeddings.ipynb`: `embeddings/multilingual-e5-small/<sample>/` with
  the S1/S2/S3 metadata and `faiss/s1_buckets*.parquet`. The exact-search buckets are used when present.
  `SAMPLE_NAME` picks the sample (e.g. `sample_500k`); empty = the most recently written one. The first
  cell lists every sample it found.
* 01's `processed/` files, for extra fields: core name, transliterated name, legal form, phonetic
  code, house number, city, state. This is optional; without it those features stay empty.

**Steps:**
1. Keep the top `CANDIDATES_PER_S1` candidates of every bucket.
2. Compute about 40 features per (S1, candidate) pair:
   * embedding similarity and rank
   * how strongly the candidate is claimed by other S1s
   * name similarity and address similarity
   * shared numbers, postcode, city and state

   Country is **not** a feature, so the model can carry over to France.
3. Train LightGBM with 5-fold cross-validation grouped by S1. Every pair gets its probability
   from a model that never saw that S1.
4. Decide the matches:
   * keep a candidate when its probability is ≥ t
   * each S2/S3 record goes to at most one S1 (the one with the highest probability)
   * t is tuned for the challenge metric (macro F0.5)
5. Check transfer between countries (train on US, score India, and the reverse) as a stand-in for France.
6. Train a final model on all pairs and save it.

**Smaller runs:** `MAX_S1` (e.g. `20000`) keeps a random subset of the sample's S1 with their
buckets, for quick experiments. The "claimed by other S1" features then see fewer competing S1, so
compare scores only between runs with the same `MAX_S1`.

**Output** (`lgbm_matcher/<model>/<sample>/`, one folder per model and sample, plus `_max<MAX_S1>` when set):

| File | Content |
|---|---|
| `lgbm_matcher.txt` | final LightGBM model |
| `matcher_config.json` | features, parameters, tuned threshold, scores |
| `oof_pairs.parquet` | every labeled pair with its out-of-fold probability and the keep decision |
| `sample_matching_results.tsv` | out-of-fold matches per labeled S1, in the challenge format |
| `metric_f05.csv`, `feature_importance.csv` | score table and feature importance |
| `unlabeled_matching_results.tsv` | matches for unlabeled S1 with buckets (e.g. France), only if there are any |

**Run on Kaggle:** attach the output of the 02 notebook (and 01's `processed/`), then Run all.
LightGBM tries the GPU and falls back to the CPU if this LightGBM build has no GPU support.
**Which e5 model's sample** (`E5_MODEL`: `small`, `base`, `large`; empty = any): needed when the same
sample exists for several models. The last cell compares the matcher scores of every model trained on
this sample.

Settings are in the first code cell and can also be set as environment variables: `E5_MODEL`, `SAMPLE_NAME`,
`MAX_S1`, `EMB_DIR`, `PROCESSED_DIR`, `MATCHER_DIR`, `CANDIDATES_PER_S1`, `LGBM_DEVICE`.

The model is trained on the sample from notebook 02, where the S1 index is much smaller than in
the full data. The features carry over, but the threshold has to be tuned again on full-size buckets.

In [ ]:
import csv
import glob
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

for pkg in ("lightgbm", "rapidfuzz"):            # both MIT
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import lightgbm as lgb
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

# ---- Which sample, and how much of it ------------------------------------------------------------------
E5_MODELS = {"small": "intfloat/multilingual-e5-small", "base": "intfloat/multilingual-e5-base",
             "large": "intfloat/multilingual-e5-large"}                  # all MIT; 118M / 278M / 560M parameters
E5_MODEL = os.environ.get("E5_MODEL", "")                          # small | base | large | model id; empty = any model
E5_MODEL = E5_MODELS.get(E5_MODEL, E5_MODEL).rstrip("/").split("/")[-1]   # the folder name 02 used, e.g. multilingual-e5-base
SAMPLE_NAME = os.environ.get("SAMPLE_NAME", "")                    # 02's sample folder, e.g. sample_500k; empty = newest
MAX_S1 = int(os.environ.get("MAX_S1", 0))                           # keep at most this many S1 (0 = all), for quick runs
# -------------------------------------------------------------------------------------------------------
CANDIDATES_PER_S1 = int(os.environ.get("CANDIDATES_PER_S1", 20))   # top candidates of each bucket to judge
N_FOLDS = 5
SEED = 42
LGBM_DEVICE = os.environ.get("LGBM_DEVICE", "auto")                # auto | cpu | gpu | cuda
CROSS_COUNTRY_CHECK = True                                          # train on one country, score the other
T0 = time.time()


def find_emb_dir():
    """Sample folder written by notebook 02 (embeddings/<model>/<sample>/ holding faiss/s1_buckets*.parquet): the one of
    model E5_MODEL named SAMPLE_NAME when set, else the most recently written one. Prints every sample found."""
    roots = [os.environ.get("EMB_DIR", ""), "embeddings", "/kaggle/working/embeddings", "/kaggle/input"]
    found = {}
    for root in roots:
        if root and os.path.isdir(root):
            for h in glob.glob(os.path.join(root, "**", "faiss", "s1_buckets*.parquet"), recursive=True):
                d = Path(h).parent.parent.resolve()
                found.setdefault(d, os.path.getmtime(h))
    if not found:
        raise FileNotFoundError("no faiss/s1_buckets*.parquet found: attach notebook 02's output or set EMB_DIR")
    print("samples found:", ", ".join(sorted(f"{d.parent.name}/{d.name}" for d in found)))
    chosen = {d: t for d, t in found.items() if (not SAMPLE_NAME or d.name == SAMPLE_NAME)
              and (not E5_MODEL or d.parent.name == E5_MODEL)}
    assert chosen, f"no sample matches E5_MODEL={E5_MODEL!r}, SAMPLE_NAME={SAMPLE_NAME!r}"
    if SAMPLE_NAME and len(chosen) > 1:
        raise ValueError(f"sample {SAMPLE_NAME!r} exists for several models {sorted(d.parent.name for d in chosen)}: "
                         "set E5_MODEL (small / base / large) to choose one")
    return max(chosen, key=chosen.get)


def find_processed_dir(manifest):
    """01's processed/ folder: $PROCESSED_DIR, the path notebook 02 used, ./processed, any Kaggle input. None if absent."""
    candidates = [os.environ.get("PROCESSED_DIR", ""), manifest.get("processed_dir", ""), "processed",
                  "/kaggle/working/processed"]
    candidates += sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/train_s1_*.parquet", recursive=True)})
    for c in candidates:
        if c and os.path.isdir(c) and glob.glob(os.path.join(c, "train_s1_*.parquet")):
            return Path(c)
    return None


EMB_DIR = find_emb_dir()
MANIFEST = json.loads((EMB_DIR / "manifest.json").read_text(encoding="utf-8")) if (EMB_DIR / "manifest.json").exists() else {}
PROCESSED_DIR = find_processed_dir(MANIFEST)
BUCKET_FILE = next(p for p in (EMB_DIR / "faiss" / "s1_buckets_exact.parquet", EMB_DIR / "faiss" / "s1_buckets.parquet")
                   if p.exists())
OUT_DIR = Path(os.environ.get("MATCHER_DIR") or ("/kaggle/working/lgbm_matcher" if os.path.isdir("/kaggle/working")
                                                 else "lgbm_matcher")) / EMB_DIR.parent.name / (EMB_DIR.name + (f"_max{MAX_S1}" if MAX_S1 else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"model: {EMB_DIR.parent.name} | sample: {EMB_DIR.name} ({MANIFEST.get('n_by_source', MANIFEST.get('n_per_source', '?'))} per source) | "
      f"S1 cap: {MAX_S1 or 'none'}")
print(f"embeddings: {EMB_DIR}\nbuckets: {BUCKET_FILE.name}\nprocessed: {PROCESSED_DIR}\noutput: {OUT_DIR}")
if PROCESSED_DIR is None:
    print("NOTE: processed/ not found, so the core-name, legal-form, phonetic, house-number, city and state "
          "features stay empty. Attach 01's output or set PROCESSED_DIR.")


def lgbm_device():
    """GPU if this LightGBM build supports it (tried on a tiny dataset), else CPU."""
    if LGBM_DEVICE != "auto":
        return LGBM_DEVICE
    if shutil.which("nvidia-smi") is None:
        return "cpu"
    x = np.random.default_rng(0).random((2000, 4))
    for device in ("cuda", "gpu"):
        try:
            lgb.train({"objective": "binary", "device_type": device, "verbose": -1},
                      lgb.Dataset(x, (x[:, 0] > 0.5).astype(int)), num_boost_round=2)
            return device
        except Exception:
            pass
    print("this LightGBM build has no GPU support, training on the CPU")
    return "cpu"


DEVICE = lgbm_device()
PARAMS = {"objective": "binary", "learning_rate": 0.1, "num_leaves": 63, "min_data_in_leaf": 100,
          "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
          "max_bin": 255 if DEVICE == "cpu" else 63, "device_type": DEVICE, "seed": SEED, "verbose": -1}
print(f"LightGBM {lgb.__version__} on {DEVICE}; {CANDIDATES_PER_S1} candidates per S1; {N_FOLDS} folds")

## Challenge metric: macro F0.5 per S1
Same as notebook 02. For every S1, F0.5 over its predicted vs true S2/S3 IDs, averaged over all S1.
An S1 with no true match scores 1.0 for an empty prediction and 0.0 otherwise.

In [ ]:
def f05(predicted, true):
    """Challenge F0.5 for one S1: predicted and true are collections of S2/S3 entity IDs."""
    predicted, true = set(predicted), set(true)
    if not true:
        return 1.0 if not predicted else 0.0
    tp = len(predicted & true)
    if tp == 0:
        return 0.0
    precision, recall = tp / len(predicted), tp / len(true)
    return 1.25 * precision * recall / (0.25 * precision + recall)


def macro_f05(pred_by_s1, true_by_s1, s1_ids):
    """Mean challenge F0.5 over s1_ids (S1 missing from pred_by_s1 = empty prediction)."""
    return float(np.mean([f05(pred_by_s1.get(s, ()), true_by_s1.get(s, ())) for s in s1_ids]))


assert round(f05(["S2-00047", "S2-00193", "S3-00812"], ["S2-00047", "S3-00812"]), 3) == 0.714
print("challenge metric OK (problem-statement example = 0.714)")

## Load the buckets, records and labels
A pair is a true match when the candidate's true S1 (from the training ground truth, stored by
notebook 02) is this S1. Only S1 from the train split are labeled.

In [ ]:
TAGS = ["tag_native_script", "tag_web_name", "tag_empty_address", "tag_honorific", "tag_landmark"]
META_COLS = ["entity_id", "split", "country", "kind", "business_name", "business_address", "name_norm", "addr_norm"] + TAGS
S1 = pd.read_parquet(EMB_DIR / "s1_meta.parquet", columns=META_COLS)
CAND = pd.concat([pd.read_parquet(EMB_DIR / f"{s}_meta.parquet", columns=META_COLS + ["true_s1"]) for s in ("s2", "s3")],
                 ignore_index=True)
P = pd.read_parquet(BUCKET_FILE, columns=["s1_entity_id", "rank", "candidate_entity_id", "candidate_source",
                                          "score", "cosine", "country"])
P = P[P["rank"] < CANDIDATES_PER_S1].sort_values(["s1_entity_id", "rank"]).reset_index(drop=True)
if MAX_S1 and P["s1_entity_id"].nunique() > MAX_S1:                # quick runs: a random subset of the S1
    keep = pd.Series(P["s1_entity_id"].unique()).sample(MAX_S1, random_state=SEED)
    P = P[P["s1_entity_id"].isin(set(keep))].reset_index(drop=True)
    print(f"MAX_S1: kept {MAX_S1:,} random S1 with their buckets")

P["labeled"] = P["s1_entity_id"].map(S1.set_index("entity_id")["split"]).eq("train").to_numpy()
P["is_true"] = P["candidate_entity_id"].map(CAND.set_index("entity_id")["true_s1"]).to_numpy() == P["s1_entity_id"].to_numpy()
TRUE_OF = CAND[CAND["kind"] == "linked"].groupby("true_s1")["entity_id"].apply(set).to_dict()   # sampled true matches

# Scored S1 (as in notebook 02): labeled singletons, and labeled S1 with at least one true match in the sample.
S1_KIND = S1.set_index("entity_id")["kind"]
bucket_s1 = pd.Series(P.loc[P["labeled"], "s1_entity_id"].unique())
EVAL_IDS = bucket_s1[bucket_s1.map(S1_KIND).eq("singleton").to_numpy() | bucket_s1.isin(TRUE_OF).to_numpy()]
N_TRUE = pd.Series([len(TRUE_OF.get(s, ())) for s in EVAL_IDS], index=EVAL_IDS.to_numpy(), name="n_true")
EVAL_COUNTRY = N_TRUE.index.map(S1.set_index("entity_id")["country"]).to_numpy()
P["s1_pos"] = P["s1_entity_id"].map(pd.Series(np.arange(len(N_TRUE)), index=N_TRUE.index)).fillna(-1).astype(int).to_numpy()

print(f"{len(P):,} pairs ({P['labeled'].sum():,} labeled, {P['is_true'].sum():,} true) for "
      f"{P['s1_entity_id'].nunique():,} S1; {len(N_TRUE):,} S1 scored ({int((N_TRUE == 0).sum()):,} singletons); "
      f"{time.time() - T0:.0f}s")
P.groupby(["country", "labeled"]).agg(pairs=("is_true", "size"), true_pairs=("is_true", "sum"),
                                      s1=("s1_entity_id", "nunique"))

## Extra fields from `processed/`
Read only the rows of the records in the buckets.

In [ ]:
EXTRA_COLS = ["core_name", "name_lat", "legal_form", "domain_stem", "phonetic_code", "street_tokens",
              "house_no", "city", "state_code"]


def read_extras(ids):
    """EXTRA_COLS for the given entity IDs from every processed parquet file (empty frame if processed/ is absent)."""
    frames = []
    if PROCESSED_DIR is not None:
        wanted = pa.array(sorted(ids), pa.large_string())
        for f in sorted(PROCESSED_DIR.glob("*_s[123]_*.parquet")):
            cols = [c for c in ["entity_id"] + EXTRA_COLS if c in pq.read_schema(f).names]
            t = pq.read_table(f, columns=cols)
            t = t.filter(pc.is_in(t["entity_id"].cast(pa.large_string()), value_set=wanted))
            if t.num_rows:
                frames.append(t.to_pandas())
    extras = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["entity_id"])
    return extras.drop_duplicates("entity_id").set_index("entity_id").reindex(columns=EXTRA_COLS)


ids = set(P["s1_entity_id"]) | set(P["candidate_entity_id"])
ENT = pd.concat([S1, CAND.drop(columns="true_s1")], ignore_index=True).drop_duplicates("entity_id")
ENT = ENT[ENT["entity_id"].isin(ids)].set_index("entity_id")
ENT = ENT.join(read_extras(ids))
for c in ["business_name", "business_address", "name_norm", "addr_norm"] + EXTRA_COLS:
    ENT[c] = ENT[c].fillna("").astype(str)
print(f"{len(ENT):,} records; share with each extra field filled:")
print((ENT[EXTRA_COLS] != "").mean().round(3).to_dict())

## Features
Per (S1, candidate) pair. Where a field is empty on either side, the feature is left empty (NaN) and
LightGBM treats it as unknown.

| Group | Features |
|---|---|
| Embedding | rank, retrieval score, cosine; gap to the S1's best candidate and to the next one; the S1's best and top-5 cosine |
| Other S1s | how many S1 buckets hold this candidate, whether this S1 is its best S1, and the score margin to its other best S1 |
| Name | edit-distance ratios, token-set/sort ratios, Jaro-Winkler, transliterated-name ratios, compact name/domain ratio, phonetic code, legal form, lengths |
| Address | ratios on the cleaned and raw address, shared street tokens, shared numbers, postcode, house number, city, state |
| Candidate flags | S2 or S3, native script, web name, empty address, honorific, landmark |

In [ ]:
try:
    from rapidfuzz.process import cpdist   # rapidfuzz >= 3.6: many pairs at once on all CPU cores
except ImportError:
    cpdist = None

RE_NUM = re.compile(r"\d+")
RE_POSTCODE = re.compile(r"(?<!\d)\d{5,6}(?!\d)")
RE_NOT_ALNUM = re.compile(r"[^0-9a-z]+")
t_feat = time.time()


def sim(scorer, a, b):
    """rapidfuzz similarity of aligned string arrays; NaN where either side is empty."""
    if cpdist is not None:
        out = cpdist(list(a), list(b), scorer=scorer, workers=-1).astype(np.float32)
    else:
        out = np.array([scorer(x, y) for x, y in zip(a, b)], np.float32)
    out[(a == "") | (b == "")] = np.nan
    return out


def same(a, b):
    """1 when equal, 0 when different, NaN when either side is empty."""
    out = (a == b).astype(np.float32)
    out[(a == "") | (b == "")] = np.nan
    return out


def set_overlap(a, b):
    """Jaccard overlap and share of a's items found in b, for aligned arrays of frozensets (NaN when empty)."""
    jac = np.full(len(a), np.nan, np.float32)
    cover = np.full(len(a), np.nan, np.float32)
    for i, (x, y) in enumerate(zip(a, b)):
        if x and y:
            inter = len(x & y)
            jac[i] = inter / len(x | y)
            cover[i] = inter / len(x)
        elif x or y:
            jac[i] = cover[i] = 0.0
    return jac, cover


# record-level helpers, computed once per record
lower_name = ENT["business_name"].str.lower()
lower_addr = ENT["business_address"].str.lower()
ENT["raw_name"], ENT["raw_addr"] = lower_name, lower_addr
ENT["compact"] = [RE_NOT_ALNUM.sub("", (d or l or n).lower())
                  for d, l, n in zip(ENT["domain_stem"], ENT["name_lat"], ENT["name_norm"])]
ENT["numbers"] = [frozenset(RE_NUM.findall(a)) for a in lower_addr]
ENT["postcode"] = [(RE_POSTCODE.findall(a) or [""])[-1] for a in lower_addr]
ENT["street_set"] = [frozenset(s.split()) for s in ENT["street_tokens"]]

QPOS = ENT.index.get_indexer(P["s1_entity_id"])
CPOS = ENT.index.get_indexer(P["candidate_entity_id"])
assert (QPOS >= 0).all() and (CPOS >= 0).all()


def pair(col):
    """Values of a record column for the S1 side and the candidate side of every pair."""
    v = ENT[col].to_numpy()
    return v[QPOS], v[CPOS]


F = pd.DataFrame(index=P.index)
# embedding and rank
F["rank"] = P["rank"].astype(np.float32)
F["score"], F["cosine"] = P["score"].astype(np.float32), P["cosine"].astype(np.float32)
by_s1 = P.groupby("s1_entity_id", sort=False)
F["s1_best_cosine"] = by_s1["cosine"].transform("max").astype(np.float32)
F["gap_to_best_score"] = (by_s1["score"].transform("max") - P["score"]).astype(np.float32)
F["gap_to_next_score"] = (P["score"] - by_s1["score"].shift(-1)).astype(np.float32)
F["s1_top5_cosine"] = P["s1_entity_id"].map(P[P["rank"] < 5].groupby("s1_entity_id")["cosine"].mean()).astype(np.float32)
# how the candidate scores against the other S1s that retrieved it
by_cand = P[["candidate_entity_id", "score"]].sort_values(["candidate_entity_id", "score"], ascending=[True, False])
by_cand["r"] = by_cand.groupby("candidate_entity_id").cumcount()
best = by_cand[by_cand["r"] == 0].set_index("candidate_entity_id")["score"]
second = by_cand[by_cand["r"] == 1].set_index("candidate_entity_id")["score"]
F["cand_n_s1"] = P.groupby("candidate_entity_id")["score"].transform("size").astype(np.float32)
F["cand_rank"] = by_cand["r"].reindex(P.index).astype(np.float32)
F["cand_margin"] = np.where(F["cand_rank"] == 0, P["score"] - P["candidate_entity_id"].map(second),
                            P["score"] - P["candidate_entity_id"].map(best)).astype(np.float32)
# name
a, b = pair("name_norm")
F["name_ratio"], F["name_token_set"] = sim(fuzz.ratio, a, b), sim(fuzz.token_set_ratio, a, b)
F["name_token_sort"], F["name_partial"] = sim(fuzz.token_sort_ratio, a, b), sim(fuzz.partial_ratio, a, b)
a, b = pair("core_name")
F["core_ratio"], F["core_jaro_winkler"] = sim(fuzz.ratio, a, b), sim(JaroWinkler.normalized_similarity, a, b)
a, b = pair("name_lat")
F["latin_ratio"], F["latin_token_set"] = sim(fuzz.ratio, a, b), sim(fuzz.token_set_ratio, a, b)
a, b = pair("raw_name")
F["raw_name_token_set"] = sim(fuzz.token_set_ratio, a, b)
a, b = pair("compact")
F["compact_ratio"], F["compact_partial"] = sim(fuzz.ratio, a, b), sim(fuzz.partial_ratio, a, b)
F["phonetic_same"] = same(*pair("phonetic_code"))
F["legal_form_same"] = same(*pair("legal_form"))
a, b = pair("name_norm")
F["s1_name_len"] = np.fromiter((len(x) for x in a), np.float32, len(a))
F["cand_name_len"] = np.fromiter((len(x) for x in b), np.float32, len(b))
# address
a, b = pair("addr_norm")
F["addr_ratio"], F["addr_token_set"] = sim(fuzz.ratio, a, b), sim(fuzz.token_set_ratio, a, b)
a, b = pair("raw_addr")
F["raw_addr_token_set"], F["raw_addr_partial"] = sim(fuzz.token_set_ratio, a, b), sim(fuzz.partial_ratio, a, b)
F["street_jaccard"], F["street_cover"] = set_overlap(*pair("street_set"))
F["numbers_jaccard"], F["numbers_cover"] = set_overlap(*pair("numbers"))
F["postcode_same"] = same(*pair("postcode"))
F["house_no_same"] = same(*pair("house_no"))
F["city_same"] = same(*pair("city"))
F["state_same"] = same(*pair("state_code"))
F["cand_addr_len"] = np.fromiter((len(x) for x in pair("raw_addr")[1]), np.float32, len(P))
# candidate flags
F["cand_is_s3"] = (P["candidate_source"] == "S3").astype(np.float32)
for tag in TAGS:
    F[f"cand_{tag[4:]}"] = pair(tag)[1].astype(np.float32)

FEATURES = list(F.columns)
print(f"{len(FEATURES)} features for {len(F):,} pairs in {time.time() - t_feat:.0f}s")
F.describe().T[["count", "mean", "min", "max"]].round(3)

## Train LightGBM with cross-validation grouped by S1
The labeled S1 are split into 5 folds. Each fold's pairs are predicted by a model trained on the
other 4 folds, which also stops training when the held-out fold stops improving.

In [ ]:
L = P[P["labeled"]].copy()
X, y = F.loc[L.index, FEATURES].to_numpy(np.float32), L["is_true"].to_numpy()
rng = np.random.default_rng(SEED)
s1_ids = L["s1_entity_id"].unique()
fold_of = pd.Series(rng.permutation(np.arange(len(s1_ids)) % N_FOLDS), index=s1_ids)
L["fold"] = L["s1_entity_id"].map(fold_of).to_numpy()


def fit(x_train, y_train, x_valid=None, y_valid=None, rounds=2000):
    """Train LightGBM; with a validation set, stop 50 rounds after its log-loss stops improving."""
    train_set = lgb.Dataset(x_train, y_train, feature_name=FEATURES)
    if x_valid is None:
        return lgb.train(PARAMS, train_set, rounds)
    valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)
    return lgb.train(PARAMS, train_set, rounds, valid_sets=[valid_set],
                     callbacks=[lgb.early_stopping(50, verbose=False)])


OOF = np.zeros(len(L), np.float32)
BEST_ITERS = []
for fold in range(N_FOLDS):
    t_fold = time.time()
    tr, va = L["fold"].to_numpy() != fold, L["fold"].to_numpy() == fold
    model = fit(X[tr], y[tr], X[va], y[va])
    OOF[va] = model.predict(X[va], num_iteration=model.best_iteration)
    BEST_ITERS.append(model.best_iteration)
    print(f"fold {fold}: {int(tr.sum()):,} train / {int(va.sum()):,} valid pairs, best iteration {model.best_iteration}, "
          f"valid log-loss {model.best_score['valid_0']['binary_logloss']:.4f}, {time.time() - t_fold:.0f}s")
L["p"] = OOF

## Decide matches and score them with the challenge metric
* **LightGBM p ≥ t:** keep every candidate whose probability is at least t.
* **+ one owner:** each S2/S3 record is kept only for the S1 that gives it the highest probability,
  because every S2/S3 has at most one true S1.
* For comparison, the notebook 02 rule (cosine ≥ t + one owner) and the ceiling (a perfect
  matcher on the same top-`CANDIDATES_PER_S1` buckets) are scored on the same S1.

t is tuned on the out-of-fold probabilities, so the scores are slightly optimistic.

In [ ]:
def score(df, keep):
    """Challenge macro F0.5 over the scored S1 when the rows of df selected by keep are the predictions."""
    pos = df["s1_pos"].to_numpy()
    sel = keep & (pos >= 0)
    n_pred = np.bincount(pos[sel], minlength=len(N_TRUE))
    tp = np.bincount(pos[sel], weights=df["is_true"].to_numpy()[sel].astype(float), minlength=len(N_TRUE))
    n_true = N_TRUE.to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        p, r = tp / n_pred, tp / n_true
        f = np.where(tp > 0, 1.25 * p * r / (0.25 * p + r), 0.0)
    f = np.where(n_true == 0, (n_pred == 0).astype(float), f)
    out = {"macro F0.5": f.mean(), "S1 with matches": f[n_true > 0].mean(), "singletons": f[n_true == 0].mean()}
    out.update({f"macro F0.5 {c}": f[EVAL_COUNTRY == c].mean() for c in sorted(set(EVAL_COUNTRY))})
    return out


def one_owner(df, p):
    """True for the rows where this S1 gives the candidate its highest p among all S1 in df."""
    order = np.argsort(-p, kind="stable")
    first = ~pd.Series(df["candidate_entity_id"].to_numpy()[order]).duplicated().to_numpy()
    owner = np.zeros(len(df), bool)
    owner[order[first]] = True
    return owner


def tune(df, p, extra, grid):
    """Best threshold on grid for keep = (p >= t) & extra, and its scores."""
    results = [(t, score(df, (p >= t) & extra)) for t in grid]
    return max(results, key=lambda r: r[1]["macro F0.5"])


GRID_P = np.round(np.arange(0.02, 0.99, 0.01), 2)
GRID_COS = np.round(np.arange(0.80, 0.995, 0.005), 3)
p, cos = L["p"].to_numpy(), L["cosine"].to_numpy()
everything = np.ones(len(L), bool)
owner_p, owner_cos = one_owner(L, p), one_owner(L, cos)

rows = [{"method": "predict nothing", "t": None, **score(L, np.zeros(len(L), bool))}]
t_cos, s_cos = tune(L, cos, owner_cos, GRID_COS)
rows.append({"method": "cosine ≥ t + one owner (notebook 02 rule)", "t": t_cos, **s_cos})
t_plain, s_plain = tune(L, p, everything, GRID_P)
rows.append({"method": "LightGBM p ≥ t", "t": t_plain, **s_plain})
T_BEST, s_best = tune(L, p, owner_p, GRID_P)
rows.append({"method": "LightGBM p ≥ t + one owner", "t": T_BEST, **s_best})
rows.append({"method": f"ceiling: perfect matcher on the top-{CANDIDATES_PER_S1} buckets", "t": None,
             **score(L, L["is_true"].to_numpy())})
METRIC = pd.DataFrame(rows)
L["kept"] = (p >= T_BEST) & owner_p

# cross-check the vectorized score against the reference metric
kept = L[L["kept"]]
pred = kept.groupby("s1_entity_id")["candidate_entity_id"].apply(list).to_dict()
reference = macro_f05(pred, TRUE_OF, N_TRUE.index)
assert abs(reference - s_best["macro F0.5"]) < 1e-9, (reference, s_best["macro F0.5"])
print(f"{len(N_TRUE):,} S1 scored; vectorized score matches the reference metric")
METRIC.round(4)

### How the threshold trades precision for recall
Macro F0.5 for different thresholds (with one owner). A flat curve around the best t means the
choice of t is not fragile.

In [ ]:
curve = pd.DataFrame([{"t": t, **score(L, (p >= t) & owner_p)} for t in np.round(np.arange(0.1, 0.95, 0.05), 2)])
curve.set_index("t")[["macro F0.5", "S1 with matches", "singletons"]].round(4)

## Transfer between countries (stand-in for France)
Train on all pairs of one country, predict the other with the threshold tuned above. If the score
stays close to the cross-validation score of that country, the model does not rely on
country-specific patterns.

In [ ]:
ROUNDS = max(50, int(np.mean(BEST_ITERS) * 1.1))
TRANSFER = []
countries = sorted(L["country"].unique())
if CROSS_COUNTRY_CHECK and len(countries) > 1:
    for test_c in countries:
        t_c = time.time()
        tr, te = L["country"].to_numpy() != test_c, L["country"].to_numpy() == test_c
        model = fit(X[tr], y[tr], rounds=ROUNDS)
        sub = L[te]
        p_te = model.predict(X[te]).astype(np.float32)
        s = score(sub, (p_te >= T_BEST) & one_owner(sub, p_te))[f"macro F0.5 {test_c}"]
        TRANSFER.append({"trained on": " + ".join(c for c in countries if c != test_c), "scored on": test_c,
                         "macro F0.5": round(s, 4),
                         "cross-validation (all countries)": round(s_best[f"macro F0.5 {test_c}"], 4),
                         "seconds": round(time.time() - t_c)})
pd.DataFrame(TRANSFER)

## Final model on all labeled pairs
Trained for the mean best iteration of the folds (+10%), then saved with its settings.

In [ ]:
t_final = time.time()
FINAL = fit(X, y, rounds=ROUNDS)
FINAL.save_model(str(OUT_DIR / "lgbm_matcher.txt"))
IMPORTANCE = pd.DataFrame({"feature": FEATURES, "gain": FINAL.feature_importance("gain"),
                           "splits": FINAL.feature_importance("split")}).sort_values("gain", ascending=False)
IMPORTANCE["gain %"] = (100 * IMPORTANCE["gain"] / IMPORTANCE["gain"].sum()).round(2)
IMPORTANCE.to_csv(OUT_DIR / "feature_importance.csv", index=False)
print(f"final model: {ROUNDS} rounds in {time.time() - t_final:.0f}s")
IMPORTANCE[["feature", "gain %", "splits"]].head(20)

## Save predictions
`sample_matching_results.tsv` has the out-of-fold matches of every labeled S1 in the challenge
format (`source1_entity_id`, comma-separated `matched_entity_ids`; empty when no match). Unlabeled
S1 that have buckets (for example France, once test S2/S3 are embedded) are predicted with the
final model.

In [ ]:
def write_matches(df, keep, path):
    """Challenge-format TSV: every S1 in df with its kept candidates (empty list when none)."""
    kept = df[keep].groupby("s1_entity_id")["candidate_entity_id"].apply(lambda s: ",".join(sorted(s)))
    out = pd.DataFrame({"source1_entity_id": df["s1_entity_id"].unique()})
    out["matched_entity_ids"] = out["source1_entity_id"].map(kept).fillna("")
    out.to_csv(path, sep="\t", index=False, quoting=csv.QUOTE_NONE)
    return len(out), int((out["matched_entity_ids"] != "").sum())


n, n_match = write_matches(L, L["kept"].to_numpy(), OUT_DIR / "sample_matching_results.tsv")
print(f"sample_matching_results.tsv: {n:,} labeled S1, {n_match:,} with at least one match")
L[["s1_entity_id", "candidate_entity_id", "country", "rank", "cosine", "fold", "p", "is_true", "kept"]].to_parquet(
    OUT_DIR / "oof_pairs.parquet", index=False)
METRIC.to_csv(OUT_DIR / "metric_f05.csv", index=False)

U = P[~P["labeled"]].copy()
if len(U):
    U["p"] = FINAL.predict(F.loc[U.index, FEATURES].to_numpy(np.float32)).astype(np.float32)
    n, n_match = write_matches(U, (U["p"].to_numpy() >= T_BEST) & one_owner(U, U["p"].to_numpy()),
                               OUT_DIR / "unlabeled_matching_results.tsv")
    print(f"unlabeled_matching_results.tsv: {n:,} unlabeled S1, {n_match:,} with at least one match")
else:
    print("no unlabeled S1 with buckets (France has no S2/S3 records in the sample), nothing to predict")

CONFIG = {"features": FEATURES, "params": PARAMS, "rounds": ROUNDS, "fold_best_iterations": BEST_ITERS,
          "threshold": T_BEST, "one_owner": True, "candidates_per_s1": CANDIDATES_PER_S1,
          "bucket_file": BUCKET_FILE.name, "embeddings": str(EMB_DIR), "processed": str(PROCESSED_DIR),
          "scores": METRIC.to_dict("records"), "transfer": TRANSFER, "seconds": round(time.time() - T0)}
(OUT_DIR / "matcher_config.json").write_text(json.dumps(CONFIG, indent=1, default=str), encoding="utf-8")
print(f"saved to {OUT_DIR} | total {time.time() - T0:.0f}s")

### Look at one S1
`show_s1(s1_entity_id)` prints the S1 record and its bucket with the out-of-fold probability.
✓ marks a true match; "kept" marks what the matcher predicts. With no argument it picks a random
S1 that has a true match in its bucket.

In [ ]:
L_BY_S1 = L.set_index("s1_entity_id")


def show_s1(s1_entity_id=None, seed=None):
    """One S1 record and its bucket, with probability, true match (✓) and kept decision."""
    if s1_entity_id is None:
        with_true = L.loc[L["is_true"], "s1_entity_id"].unique()
        s1_entity_id = np.random.default_rng(seed).choice(with_true)
    rec = ENT.loc[s1_entity_id]
    print(f"S1 {s1_entity_id} [{rec['country']}]: {rec['business_name']} | {rec['business_address']}")
    print(f"true matches in the sample: {len(TRUE_OF.get(s1_entity_id, ()))}")
    b = L_BY_S1.loc[[s1_entity_id]].sort_values("rank")
    return pd.DataFrame({"rank": b["rank"].to_numpy(), "p": b["p"].round(3).to_numpy(),
                         "cosine": b["cosine"].round(3).to_numpy(),
                         "true": np.where(b["is_true"], "✓", ""), "kept": np.where(b["kept"], "kept", ""),
                         "candidate": b["candidate_entity_id"].to_numpy(),
                         "name": ENT.loc[b["candidate_entity_id"], "business_name"].to_numpy(),
                         "address": ENT.loc[b["candidate_entity_id"], "business_address"].to_numpy()})


show_s1(seed=0)

## Compare e5 models on this sample
The matcher scores of every model whose buckets of this sample were trained here (run this notebook
once per `E5_MODEL`). The ceiling row shows how much the buckets allow; the LightGBM row what the
matcher reached.

In [ ]:
runs = {}
for f in sorted(OUT_DIR.parent.parent.glob(f"*/{OUT_DIR.name}/metric_f05.csv")):
    m = pd.read_csv(f)
    runs[f.parent.parent.name] = m.set_index("method")
if len(runs) < 2:
    print(f"only {list(runs)} trained on {OUT_DIR.name}; run this notebook with another E5_MODEL to compare")
else:
    keep = [c for c in ("macro F0.5", "macro F0.5 India", "macro F0.5 US") if c in next(iter(runs.values())).columns]
    display(pd.concat({k: v[keep] for k, v in runs.items()}, axis=1).round(4))